# Getting The Feature Vectors From Images

In this file we will implement different CNN models that convert the image to it's feature vectors. I will save each of the vector as a csv file. Maybe implement Dask for efficiency. Will use Tensorflow here

In [1]:
# Import required libraries
import os
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tqdm import tqdm
import glob

2025-10-11 18:20:14.007320: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-11 18:20:14.229009: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-11 18:20:15.675200: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Configuration parameters
IMG_SIZE = (224, 224)  # MobileNetV2 input size
BATCH_SIZE = 32
INPUT_DIR = '/home/arnavw/Documents/amazon-ml-2025/images/train_part2'
OUTPUT_CSV = '/home/arnavw/Documents/amazon-ml-2025/dataset/img_features_part2.csv'

print(f"Input directory: {INPUT_DIR}")
print(f"Output CSV file: {OUTPUT_CSV}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")

Input directory: /home/arnavw/Documents/amazon-ml-2025/images/train_part2
Output CSV file: /home/arnavw/Documents/amazon-ml-2025/dataset/img_features_part2.csv
Image size: (224, 224)
Batch size: 32


In [3]:
# Load pre-trained MobileNetV2 model
print("Loading MobileNetV2 model...")

# Load MobileNetV2 pre-trained on ImageNet, without the top classification layer
model = MobileNetV2(
    weights='imagenet',
    include_top=False,  # Remove the final classification layer
    pooling='avg',      # Add global average pooling
    input_shape=(224, 224, 3)
)

print(f"Model loaded successfully!")
print(f"Model output shape: {model.output_shape}")
print(f"Feature vector size: {model.output_shape[1]}")

# Make model non-trainable since we're only using it for feature extraction
model.trainable = False

Loading MobileNetV2 model...


I0000 00:00:1760187029.778912   36818 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6155 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model loaded successfully!
Model output shape: (None, 1280)
Feature vector size: 1280


In [4]:
# Helper functions for image processing
def load_and_preprocess_image(image_path):
    """
    Load and preprocess a single image for MobileNetV2.
    
    Args:
        image_path (str): Path to the image file
        
    Returns:
        np.ndarray: Preprocessed image array
    """
    try:
        # Load image using PIL
        image = Image.open(image_path)
        
        # Convert to RGB if needed (handles RGBA, grayscale, etc.)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Resize to target size
        image = image.resize(IMG_SIZE)
        
        # Convert to numpy array
        image_array = np.array(image)
        
        # Preprocess for MobileNetV2 (scales to [-1, 1])
        image_array = preprocess_input(image_array)
        
        return image_array
    
    except Exception as e:
        print(f"Error processing image {image_path}: {str(e)}")
        return None

def extract_features_batch(image_paths, model):
    """
    Extract features from a batch of images.
    
    Args:
        image_paths (list): List of image file paths
        model: Pre-trained model for feature extraction
        
    Returns:
        np.ndarray: Feature vectors for the batch
    """
    batch_images = []
    valid_paths = []
    
    for path in image_paths:
        processed_img = load_and_preprocess_image(path)
        if processed_img is not None:
            batch_images.append(processed_img)
            valid_paths.append(path)
    
    if not batch_images:
        return np.array([]), []
    
    # Stack images into batch
    batch_array = np.stack(batch_images)
    
    # Extract features
    features = model.predict(batch_array, verbose=0)
    
    return features, valid_paths

print("Helper functions defined successfully!")

Helper functions defined successfully!


In [5]:
# Get all image paths from the input directory
print("Scanning for images...")

# Get all image files (common formats)
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']
image_paths = []

for extension in image_extensions:
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, extension)))
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, extension.upper())))

# Sort paths for consistent ordering
image_paths.sort()

print(f"Found {len(image_paths)} images in {INPUT_DIR}")

# Display first few image names
print("\nFirst 5 images:")
for i, path in enumerate(image_paths[:5]):
    print(f"{i+1}. {os.path.basename(path)}")

if len(image_paths) > 5:
    print(f"... and {len(image_paths) - 5} more")

Scanning for images...
Found 19754 images in /home/arnavw/Documents/amazon-ml-2025/images/train_part2

First 5 images:
1. 018kdDYIAYL.jpg
2. 11+1w3qzdnL.jpg
3. 11-cLFswjlL.jpg
4. 110aIeKJdaL.jpg
5. 114IHXdZVdL.jpg
... and 19749 more


In [6]:
# Extract features from all images
print("Starting feature extraction...")

all_features = []
all_image_names = []
failed_images = []

# Process images in batches
num_batches = (len(image_paths) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in tqdm(range(num_batches), desc="Processing batches"):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(image_paths))
    
    batch_paths = image_paths[start_idx:end_idx]
    
    try:
        # Extract features for this batch
        features, valid_paths = extract_features_batch(batch_paths, model)
        
        if len(features) > 0:
            all_features.append(features)
            
            # Store image names (without full path)
            batch_names = [os.path.basename(path) for path in valid_paths]
            all_image_names.extend(batch_names)
            
            # Track failed images in this batch
            failed_in_batch = set(batch_paths) - set(valid_paths)
            failed_images.extend(failed_in_batch)
        else:
            failed_images.extend(batch_paths)
            
    except Exception as e:
        print(f"Error processing batch {batch_idx}: {str(e)}")
        failed_images.extend(batch_paths)

# Concatenate all features
if all_features:
    feature_matrix = np.vstack(all_features)
    print(f"\nFeature extraction completed!")
    print(f"Successfully processed: {len(all_image_names)} images")
    print(f"Failed to process: {len(failed_images)} images")
    print(f"Feature matrix shape: {feature_matrix.shape}")
else:
    print("No features were extracted!")

# Display failed images if any
if failed_images:
    print(f"\nFailed images ({len(failed_images)}):")
    for img in failed_images[:5]:  # Show first 5 failed images
        print(f"  - {os.path.basename(img)}")
    if len(failed_images) > 5:
        print(f"  ... and {len(failed_images) - 5} more")

Starting feature extraction...


Processing batches:   0%|          | 0/618 [00:00<?, ?it/s]2025-10-11 18:23:02.453732: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fbbe0014b00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-11 18:23:02.453759: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2025-10-11 18:23:02.485152: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-11 18:23:02.892301: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91400
2025-10-11 18:23:08.380049: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-11 18:23:08.465452: E external/local_x

Error processing image /home/arnavw/Documents/amazon-ml-2025/images/train_part2/41OUm4FmtoL.jpg: cannot identify image file '/home/arnavw/Documents/amazon-ml-2025/images/train_part2/41OUm4FmtoL.jpg'


2025-10-11 18:23:21.110202: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-11 18:23:21.194511: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
Processing batches:  19%|█▉        | 117/618 [01:03<04:13,  1.98it/s]

Error processing image /home/arnavw/Documents/amazon-ml-2025/images/train_part2/61FUSWRqStL.jpg: image file is truncated (26 bytes not processed)


...

Error processing image /home/arnavw/Documents/amazon-ml-2025/images/train_part2/81VjJjICaUL.jpg: image file is truncated (1 bytes not processed)


Processing batches:  82%|████████▏ | 504/618 [10:05<03:21,  1.77s/it]

Error processing image /home/arnavw/Documents/amazon-ml-2025/images/train_part2/81d72WNBCML.jpg: image file is truncated (2 bytes not processed)


Processing batches: 100%|█████████▉| 617/618 [13:21<00:02,  2.60s/it]2025-10-11 18:36:27.453183: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-11 18:36:27.548597: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
Processing batches: 100%|██████████| 618/618 [13:30<00:00,  1.31s/it]


Feature extraction completed!
Successfully processed: 19748 images
Failed to process: 6 images
Feature matrix shape: (19748, 1280)

Failed images (6):
  - 41OUm4FmtoL.jpg
  - 61FUSWRqStL.jpg
  - 61Ma1vH5RNL.jpg
  - 71is-1uS9RL.jpg
  - 81VjJjICaUL.jpg
  ... and 1 more


In [7]:
# Save features to CSV file
if 'feature_matrix' in locals() and len(all_image_names) > 0:
    print("Saving features to CSV...")
    
    # Create DataFrame with image names and features
    # Feature columns will be named as feature_0, feature_1, ..., feature_n
    feature_columns = [f'feature_{i}' for i in range(feature_matrix.shape[1])]
    
    # Create DataFrame
    df = pd.DataFrame(feature_matrix, columns=feature_columns)
    df.insert(0, 'image_name', all_image_names)  # Insert image name as first column
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    
    # Save to CSV
    df.to_csv(OUTPUT_CSV, index=False)
    
    print(f"Features saved to: {OUTPUT_CSV}")
    print(f"CSV shape: {df.shape}")
    print(f"Columns: image_name + {feature_matrix.shape[1]} feature columns")
    
    # Display first few rows
    print("\nFirst 3 rows of the saved data:")
    print(df.head(3))
    
    # Display basic statistics
    print(f"\nFeature statistics:")
    print(f"Mean: {feature_matrix.mean():.6f}")
    print(f"Std: {feature_matrix.std():.6f}")
    print(f"Min: {feature_matrix.min():.6f}")
    print(f"Max: {feature_matrix.max():.6f}")
    
else:
    print("No features to save!")

Saving features to CSV...
Features saved to: /home/arnavw/Documents/amazon-ml-2025/dataset/img_features_part2.csv
CSV shape: (19748, 1281)
Columns: image_name + 1280 feature columns

First 3 rows of the saved data:
        image_name  feature_0  feature_1  feature_2  feature_3  feature_4  \
0  018kdDYIAYL.jpg   0.986482   0.136997   0.170741   0.217387   0.000000   
1  11+1w3qzdnL.jpg   0.810135   0.290827   0.000000   0.094798   0.342969   
2  11-cLFswjlL.jpg   1.109655   0.219456   0.000000   0.000000   0.042501   

   feature_5  feature_6  feature_7  feature_8  ...  feature_1270  \
0   1.175086   0.006259   0.026487   0.975338  ...      1.930682   
1   0.000000   0.012480   0.024329   0.025941  ...      0.000000   
2   0.019904   0.075394   0.000000   0.025753  ...      0.000000   

   feature_1271  feature_1272  feature_1273  feature_1274  feature_1275  \
0      0.050312      0.015244      0.000000      0.000000      0.000000   
1      0.000000      0.142634      0.005916      0.09

# Summary

## Feature Extraction Results

We have successfully completed feature extraction from images in `train_part1` using a pre-trained MobileNetV2 model:

### Key Results:
- **Total Images Found**: 20,083
- **Successfully Processed**: 20,045 (99.8% success rate)
- **Failed Images**: 38 (mostly due to corrupted/truncated files)
- **Feature Vector Size**: 1,280 dimensions per image
- **Output File**: `/home/arnavw/Documents/amazon-ml-2025/dataset/img_features_part1.csv`

### Model Details:
- **Architecture**: MobileNetV2 pre-trained on ImageNet
- **Input Size**: 224×224 pixels
- **Feature Extraction**: Global Average Pooling layer output (1,280 features)
- **Preprocessing**: Images scaled to [-1, 1] range using MobileNetV2 preprocessing


The extracted features capture visual characteristics of each product image and can be used for various downstream machine learning tasks.